In [246]:
# 1. Imports and data loading

import pandas as pd
import numpy as np
from datetime import timedelta
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df_trans = pd.read_csv(r"C:\Users\user\Desktop\AI-Cohort\csv\fraud_transactions.csv")
df_cust = pd.read_csv(r"C:\Users\user\Desktop\AI-Cohort\csv\customer_profiles.csv")

df_trans["timestamp"] = pd.to_datetime(df_trans["timestamp"])

print("Transactions:", df_trans.shape)
print("Customers:", df_cust.shape)

Transactions: (12000, 8)
Customers: (1000, 5)


In [247]:
# 2. Add customer profile information

df_trans = df_trans.merge(df_cust[["customer_id", "home_location"]],on="customer_id",how="left")

print(df_trans[["customer_id", "location", "home_location"]].head())

  customer_id   location home_location
0       C0031       Pune          Pune
1       C0665        NaN     Hyderabad
2       C0422  Bengaluru     Bengaluru
3       C0107     Mumbai        Mumbai
4       C0455  Bengaluru     Bengaluru


In [248]:
customer_histories = {
    customer_id: customer_df
    for customer_id, customer_df
    in df_trans.groupby("customer_id")
}

In [ ]:
evidence_rows = []

for _, row in df_trans.iterrows():

    evidence_rows.append(
        get_evidence(
            row,
            customer_histories
        )
    )

evidence_df = pd.DataFrame(
    evidence_rows,
    index=df_trans.index
)

for column in evidence_df.columns:
    df_trans[column] = evidence_df[column]
print("\nAMOUNT")
print(
    pd.crosstab(
        df_trans["amount_unusual"],
        df_trans["fraud"],
        normalize="columns"
    )
)

print("\nLOCATION")
print(
    pd.crosstab(
        df_trans["location_unusual"],
        df_trans["fraud"],
        normalize="columns"
    )
)

print("\nMERCHANT")
print(
    pd.crosstab(
        df_trans["merchant_unusual"],
        df_trans["fraud"],
        normalize="columns"
    )
)

print("\nFREQUENCY")
print(
    pd.crosstab(
        df_trans["frequency_unusual"],
        df_trans["fraud"],
        normalize="columns"
    )
)

In [279]:
# 4. Evidence functions

def get_customer_history(customer_id, timestamp, customer_histories):
    customer = customer_histories[customer_id]
    current_time = pd.to_datetime(timestamp)

    history = customer.loc[customer["timestamp"] < current_time]

    return history.drop(columns="fraud")


def check_amount(current_amount, history):
    if history.empty:
        return False

    avg_amount = history["amount"].mean()

    if current_amount > 4 * avg_amount:
        return True

    return False


def check_location(current_location, historical_locations):
    if pd.isna(current_location):
        return None

    if current_location in historical_locations.values:
        return False

    return True


def check_merchant(current_merchant, historical_merchants):
    if pd.isna(current_merchant):
        return None

    if current_merchant in historical_merchants.values:
        return False

    return True


def check_frequency(current_timestamp, history):
    if history.empty:
        return False
    current_timestamp = pd.to_datetime(current_timestamp)
    history = history.sort_values("timestamp")
    prev_transaction = history.iloc[-1]["timestamp"]
    current_gap = current_timestamp - prev_transaction
    # print("Current Gap: ",current_gap)
    historical_gaps = history["timestamp"].diff().dropna()
    if historical_gaps.empty:
        return False
    normal_median_gap = historical_gaps.median()
    # print("Normal Median Gap ",normal_median_gap)
    frequency_threshold = normal_median_gap * 0.5
    return current_gap < frequency_threshold

def check_velocity(timestamp, history):
    if history.empty:
        return False

    recent_transactions = history[((timestamp - history["timestamp"]).dt.total_seconds() / 60) <= 30]

    return len(recent_transactions) >= 3

In [257]:
# 5. Generate evidence for every transaction

def get_evidence(transaction, customer_histories):
    history = get_customer_history(transaction["customer_id"],transaction["timestamp"],customer_histories)

    amount_unusual = check_amount(transaction["amount"],history)

    location_unusual = check_location(transaction["location"],history["location"])

    merchant_unusual = check_merchant(transaction["merchant"],history["merchant"])

    frequency_unusual = check_frequency(transaction["timestamp"],history)

    velocity_unusual = check_velocity(transaction["timestamp"],history)

    return {
        "amount_unusual": amount_unusual,
        "location_unusual": location_unusual,
        "merchant_unusual": merchant_unusual,
        "frequency_unusual": frequency_unusual,
        "velocity_unusual": velocity_unusual
    }


evidence_rows = []

for _, row in df_trans.iterrows():
    evidence_rows.append(
        get_evidence(row, customer_histories)
    )

evidence_df = pd.DataFrame(evidence_rows, index=df_trans.index)

for column in evidence_df.columns:
    df_trans[column] = evidence_df[column]

print(df_trans[
    [
        "amount_unusual",
        "location_unusual",
        "merchant_unusual",
        "frequency_unusual",
        "velocity_unusual"
    ]
].value_counts(dropna=False))

amount_unusual  location_unusual  merchant_unusual  frequency_unusual  velocity_unusual
False           False             False             False              False               5799
                                  True              False              False               2210
                True              True              False              False                956
                False             False             True               False                718
                True              False             True               False                479
True            False             False             True               False                456
False           NaN               False             False              False                279
                False             NaN               False              False                259
                                  True              True               False                249
                True              True          

In [280]:
# 6. Calculate likelihoods

def calculate_likelihoods(df_trans):
    fraud_trans = df_trans.loc[df_trans["fraud"] == 1]
    legit_trans = df_trans.loc[df_trans["fraud"] == 0]

    total_fraud = len(fraud_trans)
    total_legit = len(legit_trans)

    fraud_amount_count = (fraud_trans["amount_unusual"] == True).sum()
    legit_amount_count = (legit_trans["amount_unusual"] == True).sum()

    fraud_location_count = (fraud_trans["location_unusual"] == True).sum()
    legit_location_count = (legit_trans["location_unusual"] == True).sum()

    fraud_merchant_count = (fraud_trans["merchant_unusual"] == True).sum()
    legit_merchant_count = (legit_trans["merchant_unusual"] == True).sum()

    fraud_frequency_count = (fraud_trans["frequency_unusual"] == True).sum()
    legit_frequency_count = (legit_trans["frequency_unusual"] == True).sum()

    fraud_velocity_count = (fraud_trans["velocity_unusual"] == True).sum()
    legit_velocity_count = (legit_trans["velocity_unusual"] == True).sum()

    # Laplace smoothing
    p_amount_fraud = (fraud_amount_count + 1) / (total_fraud + 2)
    p_amount_legit = (legit_amount_count + 1) / (total_legit + 2)

    p_location_fraud = (fraud_location_count + 1) / (total_fraud + 2)
    p_location_legit = (legit_location_count + 1) / (total_legit + 2)

    p_merchant_fraud = (fraud_merchant_count + 1) / (total_fraud + 2)
    p_merchant_legit = (legit_merchant_count + 1) / (total_legit + 2)

    p_frequency_fraud = (fraud_frequency_count + 1) / (total_fraud + 2)
    p_frequency_legit = (legit_frequency_count + 1) / (total_legit + 2)

    p_velocity_fraud = (fraud_velocity_count + 1) / (total_fraud + 2)
    p_velocity_legit = (legit_velocity_count + 1) / (total_legit + 2)

    return (
        p_amount_fraud, p_amount_legit,
        p_location_fraud, p_location_legit,
        p_merchant_fraud, p_merchant_legit,
        p_frequency_fraud, p_frequency_legit,
        p_velocity_fraud, p_velocity_legit
    )


likelihoods = calculate_likelihoods(df_trans)
print(likelihoods)



(np.float64(0.2647352647352647), np.float64(9.998000399920016e-05), np.float64(0.35564435564435565), np.float64(0.10007998400319935), np.float64(0.20279720279720279), np.float64(0.33833233353329334), np.float64(0.9995004995004995), np.float64(0.028594281143771244), np.float64(0.0004995004995004995), np.float64(9.998000399920016e-05))


In [282]:
print(
    df_trans.groupby("fraud")["fraud_belief"].describe()
)
print("\nAverage fraud belief by actual class:")
print(
    df_trans.groupby("fraud")["fraud_belief"].mean()
)

         count      mean       std       min       25%       50%       75%  \
fraud                                                                        
0      10000.0  0.014291  0.084240  0.000009  0.000014  0.000017  0.000017   
1       2000.0  0.742092  0.215064  0.367236  0.538443  0.852720  0.999762   

            max  
fraud            
0      0.619666  
1      0.999952  

Average fraud belief by actual class:
fraud
0    0.014291
1    0.742092
Name: fraud_belief, dtype: float64


In [259]:
# 7. Calculate fraud belief

def calculate_fraud_belief(evidence, likelihoods):

    prior_fraud = 0.05
    prior_legit = 1 - prior_fraud

    (
        p_amount_fraud,
        p_amount_legit,
        p_location_fraud,
        p_location_legit,
        p_merchant_fraud,
        p_merchant_legit,
        p_frequency_fraud,
        p_frequency_legit,
        p_velocity_fraud,
        p_velocity_legit
    ) = likelihoods

    if evidence["amount_unusual"] == True:
        amount_fraud = p_amount_fraud
        amount_legit = p_amount_legit
    else:
        amount_fraud = 1 - p_amount_fraud
        amount_legit = 1 - p_amount_legit

    if evidence["location_unusual"] == True:
        location_fraud = p_location_fraud
        location_legit = p_location_legit
    elif evidence["location_unusual"] == False:
        location_fraud = 1 - p_location_fraud
        location_legit = 1 - p_location_legit
    else:
        location_fraud = 1
        location_legit = 1

    if evidence["merchant_unusual"] == True:
        merchant_fraud = p_merchant_fraud
        merchant_legit = p_merchant_legit
    elif evidence["merchant_unusual"] == False:
        merchant_fraud = 1 - p_merchant_fraud
        merchant_legit = 1 - p_merchant_legit
    else:
        merchant_fraud = 1
        merchant_legit = 1

    if evidence["frequency_unusual"] == True:
        frequency_fraud = p_frequency_fraud
        frequency_legit = p_frequency_legit
    else:
        frequency_fraud = 1 - p_frequency_fraud
        frequency_legit = 1 - p_frequency_legit

    if evidence["velocity_unusual"] == True:
        velocity_fraud = p_velocity_fraud
        velocity_legit = p_velocity_legit
    else:
        velocity_fraud = 1 - p_velocity_fraud
        velocity_legit = 1 - p_velocity_legit

    fraud_probability = (prior_fraud * amount_fraud * location_fraud * merchant_fraud * frequency_fraud * velocity_fraud)

    legit_probability = (prior_legit  * amount_legit * location_legit * merchant_legit * frequency_legit * velocity_legit)

    return fraud_probability / (fraud_probability + legit_probability)

In [260]:
# 8. Cost model and action selection

def calculate_action_costs(fraud_belief):
    legit_belief = 1 - fraud_belief

    approve_cost = fraud_belief * 100
    question_cost = (fraud_belief * 10 + legit_belief * 2)
    examine_cost = (fraud_belief * 5  + legit_belief * 8)
    decline_cost = legit_belief * 20

    return {
        "Approve": approve_cost,
        "Question": question_cost,
        "Examine": examine_cost,
        "Decline": decline_cost
    }


def choose_action(costs):
    return min(costs, key=costs.get)

In [261]:
# 9. Missing-information handling

def get_questions(evidence):
    questions = []

    # Ask customer to confirm the transaction
    if evidence["amount_unusual"] == True or evidence["frequency_unusual"] == True:
        questions.append("transaction_confirmation")

    # Ask for missing location
    if evidence["location_unusual"] is None:
        questions.append("location")

    return questions


def update_transaction(transaction, answers):
    transaction = transaction.copy()

    for question, answer in answers.items():
        if answer is not None:
            transaction[question] = answer

    return transaction


def simulate_customer_response(transaction, questions):
    answers = {}

    for question in questions:
        ####### Need Human in Loop tp confirm whether the transaction was made by person or not
        if question == "transaction_confirmation":
            if transaction["fraud"] == 0:
                answers["transaction_confirmation"] = True
            else:
                answers["transaction_confirmation"] = False
        elif question == "location":
            answers["location"] = transaction["home_location"]

        elif question == "merchant":
            # Keep None to simulate an unanswered merchant question.
            answers["merchant"] = None

    return answers
def handle_transaction_confirmation(answer):
    if answer == True:
        return "Approve"

    elif answer == False:
        return "Decline"

    else:
        return "Examine"
def execute_action(action, transaction, evidence):

    if action == "Approve":
        return "Approve"

    elif action == "Decline":
        return "Decline"

    elif action == "Examine":
        return "Examine"

    elif action == "Question":

        questions = get_questions(evidence)

        answers = simulate_customer_response(
            transaction,
            questions
        )

        if "transaction_confirmation" in answers:
            return handle_transaction_confirmation(
                answers["transaction_confirmation"]
            )

        return "Examine"
test_transaction = df_trans.iloc[2000]


In [262]:
result = process_transaction(
    fraud_transaction,
    likelihoods
)

print(result)

(np.float64(0.5384430362492056), 'Question', 'Decline')


In [263]:
def make_decision(evidence, transaction, likelihoods):

    fraud_belief = calculate_fraud_belief(
        evidence,
        likelihoods
    )

    costs = calculate_action_costs(fraud_belief)

    action = choose_action(costs)
    # If the agent chooses Question
    if action == "Question":

        questions = get_questions(evidence)

        answers = simulate_customer_response(
            transaction,
            questions
        )

        # Customer confirmed the transaction
        if "transaction_confirmation" in answers:
            final_action = handle_transaction_confirmation(
                answers["transaction_confirmation"]
            )

            return fraud_belief, costs, action, final_action
    return fraud_belief, costs, action, action

def process_transaction(transaction, likelihoods):

    evidence = {
        "amount_unusual": transaction["amount_unusual"],
        "location_unusual": transaction["location_unusual"],
        "merchant_unusual": transaction["merchant_unusual"],
        "frequency_unusual": transaction["frequency_unusual"],
        "velocity_unusual": transaction["velocity_unusual"]
    }

    fraud_belief, costs, initial_action, final_action = make_decision(
        evidence,
        transaction,
        likelihoods
    )

    return fraud_belief, initial_action, final_action
result = process_transaction(
    test_transaction,
    likelihoods
)

print(result)

(np.float64(1.7160836050374996e-05), 'Approve', 'Approve')


In [264]:
print("*"*10,"Initial Action","*"*10)
print(results_df["initial_action"].value_counts())
print("*"*10,"Final Action","*"*10)
print(results_df["final_action"].value_counts())

********** Initial Action **********
initial_action
Question    9895
Approve     1633
Decline      472
Name: count, dtype: int64
********** Final Action **********
final_action
Approve     6017
Question    4672
Decline     1311
Name: count, dtype: int64


In [285]:
# 12. Run the agent on the complete dataset

results = []

for _, transaction in df_trans.iterrows():

    fraud_belief, initial_action, final_action = process_transaction(
        transaction,
        likelihoods
    )

    results.append({
        "transaction_id": transaction["transaction_id"],
        "fraud_belief": fraud_belief,
        "initial_action": initial_action,
        "final_action": final_action,
        "actual_fraud": transaction["fraud"]
    })

results_df = pd.DataFrame(results)

print(results_df.shape)

(12000, 5)


In [286]:
print(results_df.head())

  transaction_id  fraud_belief initial_action final_action  actual_fraud
0         T00301      0.000042        Approve      Approve             0
1         T06641      0.000012        Approve      Approve             0
2         T04211      0.000042        Approve      Approve             0
3         T01061      0.000042        Approve      Approve             0
4         T04541      0.000042        Approve      Approve             0


In [287]:
print("\nINITIAL ACTIONS")
print(results_df["initial_action"].value_counts())

print("\nFINAL ACTIONS")
print(results_df["final_action"].value_counts())


INITIAL ACTIONS
initial_action
Approve     9715
Decline     1213
Question    1021
Examine       51
Name: count, dtype: int64

FINAL ACTIONS
final_action
Approve    9991
Decline    1958
Examine      51
Name: count, dtype: int64


In [288]:
print("\nFINAL ACTION BY ACTUAL STATE")
print(
    pd.crosstab(
        results_df["actual_fraud"],
        results_df["final_action"]
    )
)


FINAL ACTION BY ACTUAL STATE
final_action  Approve  Decline  Examine
actual_fraud                           
0                9991        0        9
1                   0     1958       42


In [ ]:
#### print("\nEXAMINED TRANSACTIONS")
print(
    results_df[results_df["final_action"] == "Examine"][
        ["transaction_id", "fraud_belief", "actual_fraud"]
    ].sort_values("fraud_belief")
)

In [266]:
pd.crosstab(results_df["final_action"],results_df["actual_fraud"])

actual_fraud,0,1
final_action,,
Approve,9991,0
Decline,0,1958
Examine,9,42


In [267]:
# Fraud detection metrics

fraud_total = (results_df["actual_fraud"] == 1).sum()

fraud_approved = ((results_df["actual_fraud"] == 1) &(results_df["final_action"] == "Approve")).sum()

fraud_examined = ((results_df["actual_fraud"] == 1) & (results_df["final_action"] == "Examine")).sum()

fraud_declined = ((results_df["actual_fraud"] == 1) &(results_df["final_action"] == "Decline")).sum()

fraud_intercepted = fraud_examined + fraud_declined

print("Total fraud:", fraud_total)
print("Fraud approved:", fraud_approved)
print("Fraud examined:", fraud_examined)
print("Fraud declined:", fraud_declined)
print("Fraud intercepted:", fraud_intercepted)

print("Fraud interception rate:",fraud_intercepted / fraud_total)

Total fraud: 2000
Fraud approved: 0
Fraud examined: 42
Fraud declined: 1958
Fraud intercepted: 2000
Fraud interception rate: 1.0


In [268]:
########################## Caculating Customer Impact ###################
legit_total = (results_df["actual_fraud"] == 0).sum()

legit_approved = ((results_df["actual_fraud"] == 0) &(results_df["final_action"] == "Approve")).sum()

legit_examined = ((results_df["actual_fraud"] == 0) & (results_df["final_action"] == "Examine")).sum()

legit_declined = ((results_df["actual_fraud"] == 0) &(results_df["final_action"] == "Decline")).sum()

print("Total legitimate:", legit_total)
print("Legitimate approved:", legit_approved)
print("Legitimate examined:", legit_examined)
print("Legitimate declined:", legit_declined)

print("Legitimate approval rate:",legit_approved / legit_total)

Total legitimate: 10000
Legitimate approved: 9991
Legitimate examined: 9
Legitimate declined: 0
Legitimate approval rate: 0.9991


In [269]:
def calculate_final_cost(row):
    if row["final_action"] == "Approve":                        ### Fraud + Approve = 100, Legit + approve = 0
        return 100 if row["actual_fraud"] == 1 else 0

    elif row["final_action"] == "Examine":                      #### Fraud + Examine = 5. Legit + examine = 8
        return 5 if row["actual_fraud"] == 1 else 8

    elif row["final_action"] == "Decline":                      ###### Fraud + Decline = 0 , Legit + decline = 20
        return 0 if row["actual_fraud"] == 1 else 20

    return 0


results_df["final_cost"] = results_df.apply(calculate_final_cost,axis=1)

print("Total actual cost:", results_df["final_cost"].sum())
print("Average cost per transaction:", results_df["final_cost"].mean())

Total actual cost: 282
Average cost per transaction: 0.0235


In [270]:
question_count = (
    results_df["initial_action"] == "Question").sum()

print("Transactions questioned:", question_count)

Transactions questioned: 1021


In [271]:
question_results = results_df[
    results_df["initial_action"] == "Question"
]

print(question_results[["initial_action", "final_action", "actual_fraud"]].value_counts())

initial_action  final_action  actual_fraud
Question        Decline       1               745
                Approve       0               276
Name: count, dtype: int64


In [272]:
question_cost = 0
for _,row in question_results.iterrows():
    if row["actual_fraud"] == 1:
        question_cost += 10
    else:
        question_cost += 5
print("Total cost in Question",question_cost)

Total cost in Question 8830


In [274]:
print(
    pd.crosstab(
        baseline_df["baseline_action"],
        baseline_df["actual_fraud"]
    )
)

actual_fraud        0     1
baseline_action            
Approve          9715     0
Decline             0  1213
Examine           285   787


In [283]:
print(
    df_trans.groupby("fraud")["fraud_belief"].describe()
)

         count      mean       std       min       25%       50%       75%  \
fraud                                                                        
0      10000.0  0.014291  0.084240  0.000009  0.000014  0.000017  0.000017   
1       2000.0  0.742092  0.215064  0.367236  0.538443  0.852720  0.999762   

            max  
fraud            
0      0.619666  
1      0.999952  


In [284]:
print("\nAverage fraud belief by actual class:")
print(
    df_trans.groupby("fraud")["fraud_belief"].mean()
)


Average fraud belief by actual class:
fraud
0    0.014291
1    0.742092
Name: fraud_belief, dtype: float64


In [290]:
# Creating 40 case test set
test_df = df_trans.sample(n=40,random_state=42).copy()

print(test_df["fraud"].value_counts())

fraud
0    29
1    11
Name: count, dtype: int64


In [291]:
print(
    test_df[
        [
            "transaction_id",
            "fraud",
            "amount_unusual",
            "location_unusual",
            "merchant_unusual",
            "frequency_unusual",
            "velocity_unusual",
            "fraud_belief"
        ]
    ].to_string(index=False)
)

transaction_id  fraud  amount_unusual location_unusual merchant_unusual  frequency_unusual  velocity_unusual  fraud_belief
        T00542      0           False            False             True              False             False      0.000009
        T09657      0           False            False            False              False             False      0.000017
        T09232      0           False            False             True              False             False      0.000009
   MERCH_C0142      1           False             True             True               True             False      0.742294
        T07241      0           False             True             True              False             False      0.000042
     LOC_C0670      1           False             True            False               True             False      0.852720
        T05856      0           False            False            False              False             False      0.000017
        T06439  

In [292]:
test_results = []

for _, transaction in test_df.iterrows():

    fraud_belief, initial_action, final_action = process_transaction(
        transaction,
        likelihoods
    )

    test_results.append({
        "transaction_id": transaction["transaction_id"],
        "fraud_belief": fraud_belief,
        "initial_action": initial_action,
        "final_action": final_action,
        "actual_fraud": transaction["fraud"]
    })

test_results_df = pd.DataFrame(test_results)

print(test_results_df)

   transaction_id  fraud_belief initial_action final_action  actual_fraud
0          T00542      0.000009        Approve      Approve             0
1          T09657      0.000017        Approve      Approve             0
2          T09232      0.000009        Approve      Approve             0
3     MERCH_C0142      0.742294        Decline      Decline             1
4          T07241      0.000042        Approve      Approve             0
5       LOC_C0670      0.852720        Decline      Decline             1
6          T05856      0.000017        Approve      Approve             0
7          T06439      0.000024        Approve      Approve             0
8          T08283      0.000024        Approve      Approve             0
9          T05555      0.000017        Approve      Approve             0
10         T02104      0.000017        Approve      Approve             0
11         T00772      0.000012        Approve      Approve             0
12         T06421      0.000042       

In [293]:
print("\nFINAL ACTION BY ACTUAL STATE")
print(
    pd.crosstab(
        test_results_df["actual_fraud"],
        test_results_df["final_action"]
    )
)


FINAL ACTION BY ACTUAL STATE
final_action  Approve  Decline
actual_fraud                  
0                  29        0
1                   0       11


In [294]:
print(test_results_df["initial_action"].value_counts())

initial_action
Approve     29
Decline      7
Question     4
Name: count, dtype: int64


In [295]:
def choose_action_threshold(fraud_belief):

    if fraud_belief < 0.20:
        return "Approve"

    elif fraud_belief < 0.50:
        return "Examine"

    elif fraud_belief < 0.80:
        return "Question"

    else:
        return "Decline"

In [296]:
policy_b_results = []

for _, transaction in test_df.iterrows():

    belief = transaction["fraud_belief"]

    initial_action = choose_action_threshold(belief)

    final_action = initial_action

    if initial_action == "Question":

        questions = get_questions(
            {
                "amount_unusual": transaction["amount_unusual"],
                "location_unusual": transaction["location_unusual"],
                "merchant_unusual": transaction["merchant_unusual"],
                "frequency_unusual": transaction["frequency_unusual"],
                "velocity_unusual": transaction["velocity_unusual"]
            }
        )

        answers = simulate_customer_response(
            transaction,
            questions
        )

        if "transaction_confirmation" in answers:
            final_action = handle_transaction_confirmation(
                answers["transaction_confirmation"]
            )

    policy_b_results.append({
        "transaction_id": transaction["transaction_id"],
        "fraud_belief": belief,
        "initial_action": initial_action,
        "final_action": final_action,
        "actual_fraud": transaction["fraud"]
    })

policy_b_results_df = pd.DataFrame(policy_b_results)

In [297]:
print("\nPOLICY B — INITIAL ACTIONS")
print(
    policy_b_results_df["initial_action"].value_counts()
)

print("\nPOLICY B — FINAL ACTIONS")
print(
    policy_b_results_df["final_action"].value_counts()
)


POLICY B — INITIAL ACTIONS
initial_action
Approve     29
Question     6
Decline      4
Examine      1
Name: count, dtype: int64

POLICY B — FINAL ACTIONS
final_action
Approve    29
Decline    10
Examine     1
Name: count, dtype: int64


In [298]:
belief = transaction["fraud_belief"]

In [299]:
def calculate_interaction_metrics(results_df):

    total = len(results_df)

    question_rate = (
        (results_df["initial_action"] == "Question").sum()
        / total
    )

    examine_rate = (
        (results_df["initial_action"] == "Examine").sum()
        / total
    )

    interaction_rate = question_rate + examine_rate

    return {
        "Question rate": question_rate,
        "Examine rate": examine_rate,
        "Interaction rate": interaction_rate
    }

In [300]:
print("Policy A:")
print(calculate_interaction_metrics(test_results_df))

print("\nPolicy B:")
print(calculate_interaction_metrics(policy_b_results_df))

Policy A:
{'Question rate': np.float64(0.1), 'Examine rate': np.float64(0.0), 'Interaction rate': np.float64(0.1)}

Policy B:
{'Question rate': np.float64(0.15), 'Examine rate': np.float64(0.025), 'Interaction rate': np.float64(0.175)}


In [301]:
print("POLICY A")
print(
    pd.crosstab(
        test_results_df["actual_fraud"],
        test_results_df["final_action"]
    )
)

print("\nPOLICY B")
print(
    pd.crosstab(
        policy_b_results_df["actual_fraud"],
        policy_b_results_df["final_action"]
    )
)

POLICY A
final_action  Approve  Decline
actual_fraud                  
0                  29        0
1                   0       11

POLICY B
final_action  Approve  Decline  Examine
actual_fraud                           
0                  29        0        0
1                   0       10        1


In [304]:
from sklearn.model_selection import train_test_split

train_df ,test_df = train_test_split(df_trans,test_size = 0.20,random_state = 42,stratify=df_trans["fraud"])
print("Training transactions:",len(train_df))
print("Test transactions:",len(test_df))

print("*"*20)
print("Training fraud distribution")
print(train_df["fraud"].value_counts())

print("*"*20)
print("Test fraud distribution")
print(test_df["fraud"].value_counts())

Training transactions: 9600
Test transactions: 2400
********************
Training fraud distribution
fraud
0    8000
1    1600
Name: count, dtype: int64
********************
Test fraud distribution
fraud
0    2000
1     400
Name: count, dtype: int64


In [306]:
# New - Calculate likelihoods

def calculate_likelihoods(df):
    fraud_trans = df_trans.loc[df_trans["fraud"] == 1]
    legit_trans = df_trans.loc[df_trans["fraud"] == 0]

    total_fraud = len(fraud_trans)
    total_legit = len(legit_trans)

    fraud_amount_count = (fraud_trans["amount_unusual"] == True).sum()
    legit_amount_count = (legit_trans["amount_unusual"] == True).sum()

    fraud_location_count = (fraud_trans["location_unusual"] == True).sum()
    legit_location_count = (legit_trans["location_unusual"] == True).sum()

    fraud_merchant_count = (fraud_trans["merchant_unusual"] == True).sum()
    legit_merchant_count = (legit_trans["merchant_unusual"] == True).sum()

    fraud_frequency_count = (fraud_trans["frequency_unusual"] == True).sum()
    legit_frequency_count = (legit_trans["frequency_unusual"] == True).sum()

    fraud_velocity_count = (fraud_trans["velocity_unusual"] == True).sum()
    legit_velocity_count = (legit_trans["velocity_unusual"] == True).sum()

    # Laplace smoothing
    p_amount_fraud = (fraud_amount_count + 1) / (total_fraud + 2)
    p_amount_legit = (legit_amount_count + 1) / (total_legit + 2)

    p_location_fraud = (fraud_location_count + 1) / (total_fraud + 2)
    p_location_legit = (legit_location_count + 1) / (total_legit + 2)

    p_merchant_fraud = (fraud_merchant_count + 1) / (total_fraud + 2)
    p_merchant_legit = (legit_merchant_count + 1) / (total_legit + 2)

    p_frequency_fraud = (fraud_frequency_count + 1) / (total_fraud + 2)
    p_frequency_legit = (legit_frequency_count + 1) / (total_legit + 2)

    p_velocity_fraud = (fraud_velocity_count + 1) / (total_fraud + 2)
    p_velocity_legit = (legit_velocity_count + 1) / (total_legit + 2)

    return (
        p_amount_fraud, p_amount_legit,
        p_location_fraud, p_location_legit,
        p_merchant_fraud, p_merchant_legit,
        p_frequency_fraud, p_frequency_legit,
        p_velocity_fraud, p_velocity_legit
    )


likelihoods = calculate_likelihoods(train_df)
print(likelihoods)



(np.float64(0.2647352647352647), np.float64(9.998000399920016e-05), np.float64(0.35564435564435565), np.float64(0.10007998400319935), np.float64(0.20279720279720279), np.float64(0.33833233353329334), np.float64(0.9995004995004995), np.float64(0.028594281143771244), np.float64(0.0004995004995004995), np.float64(9.998000399920016e-05))


In [307]:
def calculate_test_belief(transaction, likelihoods):

    evidence = {
        "amount_unusual": transaction["amount_unusual"],
        "location_unusual": transaction["location_unusual"],
        "merchant_unusual": transaction["merchant_unusual"],
        "frequency_unusual": transaction["frequency_unusual"],
        "velocity_unusual": transaction["velocity_unusual"]
    }

    return calculate_fraud_belief(
        evidence,
        likelihoods
    )


test_df["fraud_belief_new"] = test_df.apply(
    lambda row: calculate_test_belief(row, likelihoods),
    axis=1
)

print(
    test_df["fraud_belief_new"].describe()
)

count    2400.000000
mean        0.135353
std         0.295013
min         0.000009
25%         0.000017
50%         0.000017
75%         0.000042
max         0.999952
Name: fraud_belief_new, dtype: float64


In [308]:
test_results = []

for _, transaction in test_df.iterrows():

    fraud_belief = transaction["fraud_belief_new"]

    costs = calculate_action_costs(fraud_belief)
    initial_action = choose_action(costs)

    final_action = initial_action

    if initial_action == "Question":

        evidence = {
            "amount_unusual": transaction["amount_unusual"],
            "location_unusual": transaction["location_unusual"],
            "merchant_unusual": transaction["merchant_unusual"],
            "frequency_unusual": transaction["frequency_unusual"],
            "velocity_unusual": transaction["velocity_unusual"]
        }

        questions = get_questions(evidence)

        answers = simulate_customer_response(
            transaction,
            questions
        )

        if "transaction_confirmation" in answers:
            final_action = handle_transaction_confirmation(
                answers["transaction_confirmation"]
            )

    test_results.append({
        "transaction_id": transaction["transaction_id"],
        "fraud_belief": fraud_belief,
        "initial_action": initial_action,
        "final_action": final_action,
        "actual_fraud": transaction["fraud"]
    })

test_results_df = pd.DataFrame(test_results)

print(test_results_df.head())

  transaction_id  fraud_belief initial_action final_action  actual_fraud
0         T07764      0.000009        Approve      Approve             0
1         T07830      0.000009        Approve      Approve             0
2    MERCH_C0701      0.742294        Decline      Decline             1
3         T08642      0.000017        Approve      Approve             0
4         T01313      0.000017        Approve      Approve             0


In [309]:
print("\nINITIAL ACTIONS")
print(test_results_df["initial_action"].value_counts())

print("\nFINAL ACTIONS")
print(test_results_df["final_action"].value_counts())


INITIAL ACTIONS
initial_action
Approve     1945
Decline      239
Question     206
Examine       10
Name: count, dtype: int64

FINAL ACTIONS
final_action
Approve    1999
Decline     391
Examine      10
Name: count, dtype: int64


In [310]:
print("\nFINAL ACTION BY ACTUAL STATE")
print(
    pd.crosstab(
        test_results_df["actual_fraud"],
        test_results_df["final_action"]
    )
)


FINAL ACTION BY ACTUAL STATE
final_action  Approve  Decline  Examine
actual_fraud                           
0                1999        0        1
1                   0      391        9


In [311]:
#################### Legitimate transaction which was examined###############
print(
    test_results_df[
        (test_results_df["actual_fraud"] == 0) &
        (test_results_df["final_action"] == "Examine")
    ][
        [
            "transaction_id",
            "fraud_belief",
            "initial_action",
            "final_action"
        ]
    ]
)

   transaction_id  fraud_belief initial_action final_action
51         T01589      0.619666        Examine      Examine


In [312]:
print(
    test_results_df[
        (test_results_df["actual_fraud"] == 1) &
        (test_results_df["final_action"] == "Examine")
    ][
        [
            "transaction_id",
            "fraud_belief",
            "initial_action",
            "final_action"
        ]
    ]
)

     transaction_id  fraud_belief initial_action final_action
501       LOC_C0997      0.619666        Examine      Examine
532     MERCH_C0718      0.574879        Examine      Examine
655      FREQ_C0834      0.619666        Examine      Examine
748      FREQ_C0467      0.619666        Examine      Examine
749       LOC_C0622      0.619666        Examine      Examine
834      FREQ_C0818      0.619666        Examine      Examine
916      FREQ_C0429      0.619666        Examine      Examine
1250     FREQ_C0621      0.619666        Examine      Examine
2312      LOC_C0719      0.619666        Examine      Examine


In [313]:
from sklearn.metrics import precision_score, recall_score, confusion_matrix

y_true = test_results_df["actual_fraud"]

y_pred = (
    test_results_df["final_action"] == "Decline"
).astype(int)

print("Precision-every transaction automatically declined was actually fraud:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Precision: 1.0
Recall: 0.9775

Confusion Matrix:
[[2000    0]
 [   9  391]]


In [314]:
total_legit = (test_results_df["actual_fraud"] == 0).sum()

legit_approved = (
    (test_results_df["actual_fraud"] == 0) &
    (test_results_df["final_action"] == "Approve")
).sum()

legit_examine = (
    (test_results_df["actual_fraud"] == 0) &
    (test_results_df["final_action"] == "Examine")
).sum()

question_count = (
    test_results_df["initial_action"] == "Question"
).sum()

examine_count = (
    test_results_df["initial_action"] == "Examine"
).sum()

print("Legitimate approval rate:",
      legit_approved / total_legit)

print("Legitimate examine rate:",
      legit_examine / total_legit)

print("Question rate:",
      question_count / len(test_results_df))

print("Examine rate:",
      examine_count / len(test_results_df))

print("Interaction rate:",
      (question_count + examine_count) / len(test_results_df))

Legitimate approval rate: 0.9995
Legitimate examine rate: 0.0005
Question rate: 0.08583333333333333
Examine rate: 0.004166666666666667
Interaction rate: 0.09


In [315]:
################# Threshold based ###############
policy_b_results = []

for _, transaction in test_df.iterrows():

    belief = transaction["fraud_belief_new"]

    initial_action = choose_action_threshold(belief)

    final_action = initial_action

    if initial_action == "Question":

        evidence = {
            "amount_unusual": transaction["amount_unusual"],
            "location_unusual": transaction["location_unusual"],
            "merchant_unusual": transaction["merchant_unusual"],
            "frequency_unusual": transaction["frequency_unusual"],
            "velocity_unusual": transaction["velocity_unusual"]
        }

        questions = get_questions(evidence)

        answers = simulate_customer_response(
            transaction,
            questions
        )

        if "transaction_confirmation" in answers:
            final_action = handle_transaction_confirmation(
                answers["transaction_confirmation"]
            )

    policy_b_results.append({
        "transaction_id": transaction["transaction_id"],
        "fraud_belief": belief,
        "initial_action": initial_action,
        "final_action": final_action,
        "actual_fraud": transaction["fraud"]
    })

policy_b_results_df = pd.DataFrame(policy_b_results)

In [316]:
print("\nPOLICY B — INITIAL ACTIONS")
print(
    policy_b_results_df["initial_action"].value_counts()
)

print("\nPOLICY B — FINAL ACTIONS")
print(
    policy_b_results_df["final_action"].value_counts()
)


POLICY B — INITIAL ACTIONS
initial_action
Approve     1945
Question     205
Decline      198
Examine       52
Name: count, dtype: int64

POLICY B — FINAL ACTIONS
final_action
Approve    1987
Decline     361
Examine      52
Name: count, dtype: int64


In [317]:
print("\nPOLICY B — FINAL ACTION BY ACTUAL STATE")

print(
    pd.crosstab(
        policy_b_results_df["actual_fraud"],
        policy_b_results_df["final_action"]
    )
)


POLICY B — FINAL ACTION BY ACTUAL STATE
final_action  Approve  Decline  Examine
actual_fraud                           
0                1987        0       13
1                   0      361       39


In [318]:
def calculate_customer_metrics(results_df):

    legit = results_df[results_df["actual_fraud"] == 0]

    metrics = {
        "Legitimate approval rate":
            (legit["final_action"] == "Approve").mean(),

        "Legitimate question rate":
            (legit["initial_action"] == "Question").mean(),

        "Legitimate examine rate":
            (legit["final_action"] == "Examine").mean(),

        "Legitimate decline rate":
            (legit["final_action"] == "Decline").mean(),

        "Overall interaction rate":
            (results_df["initial_action"] == "Question").mean()
    }

    return metrics


print("POLICY A — CUSTOMER METRICS")
print(calculate_customer_metrics(test_results_df))

print("\nPOLICY B — CUSTOMER METRICS")
print(calculate_customer_metrics(policy_b_results_df))

POLICY A — CUSTOMER METRICS
{'Legitimate approval rate': np.float64(0.9995), 'Legitimate question rate': np.float64(0.027), 'Legitimate examine rate': np.float64(0.0005), 'Legitimate decline rate': np.float64(0.0), 'Overall interaction rate': np.float64(0.08583333333333333)}

POLICY B — CUSTOMER METRICS
{'Legitimate approval rate': np.float64(0.9935), 'Legitimate question rate': np.float64(0.021), 'Legitimate examine rate': np.float64(0.0065), 'Legitimate decline rate': np.float64(0.0), 'Overall interaction rate': np.float64(0.08541666666666667)}


In [319]:
examined_b = policy_b_results_df[
    policy_b_results_df["initial_action"] == "Examine"
]

print("Number of examined transactions:", len(examined_b))

print("\nActual fraud distribution:")
print(examined_b["actual_fraud"].value_counts())

print("\nExamined transactions:")
print(
    examined_b[
        [
            "transaction_id",
            "fraud_belief",
            "actual_fraud",
            "final_action"
        ]
    ].sort_values("fraud_belief", ascending=False)
)

Number of examined transactions: 52

Actual fraud distribution:
actual_fraud
1    39
0    13
Name: count, dtype: int64

Examined transactions:
     transaction_id  fraud_belief  actual_fraud final_action
1265    MERCH_C0917      0.491933             1      Examine
1315         T05474      0.491933             0      Examine
2368    MERCH_C0262      0.491933             1      Examine
270      FREQ_C0896      0.491933             1      Examine
1645     FREQ_C0454      0.491933             1      Examine
1094     FREQ_C0840      0.491933             1      Examine
1863         T01974      0.491933             0      Examine
919      FREQ_C0837      0.491933             1      Examine
890     MERCH_C0188      0.491933             1      Examine
1173    MERCH_C0081      0.447683             1      Examine
156     MERCH_C0415      0.447683             1      Examine
1892         T09028      0.447683             0      Examine
2371    MERCH_C0164      0.447683             1      Examine
197

In [320]:
############### Enhancing Examine Action ####################
def examine_transaction(transaction, history):
    
    recent_transactions = history[
        (transaction["timestamp"] - history["timestamp"]).dt.total_seconds() / 3600 <= 24
    ]
    
    transactions_last_24h = len(recent_transactions)
    
    return {
        "transactions_last_24h": transactions_last_24h
    }

In [322]:
# Pick one transaction from the test set
transaction = test_df.iloc[0]

# Get that customer's history
history = get_customer_history(
    transaction["customer_id"],
    transaction["timestamp"],
    customer_histories
)

# Run the examination
examination_result = examine_transaction(
    transaction,
    history
)

print(examination_result)

{'transactions_last_24h': 0}


In [323]:
# Find transactions where the examination finds at least 1
recent_activity_cases = []

for _, transaction in test_df.iterrows():

    history = get_customer_history(
        transaction["customer_id"],
        transaction["timestamp"],
        customer_histories
    )

    result = examine_transaction(transaction, history)

    if result["transactions_last_24h"] > 0:
        recent_activity_cases.append(
            (
                transaction["transaction_id"],
                transaction["fraud"],
                result["transactions_last_24h"]
            )
        )

print(recent_activity_cases[:10])


[('T07830', 0, 1), ('MERCH_C0701', 1, 3), ('T08642', 0, 1), ('T00742', 0, 1), ('FREQ_C0577', 1, 1), ('T01338', 0, 1), ('FREQ_C0677', 1, 1), ('T00812', 0, 1), ('T00529', 0, 1), ('T01514', 0, 1)]


In [324]:
def apply_examination(evidence, examination_result):

    if examination_result["transactions_last_24h"] >= 3:
        evidence["frequency_unusual"] = True

    return evidence

In [325]:
apply_examination(evidence, examination_result)

{'amount_unusual': False,
 'location_unusual': False,
 'merchant_unusual': False,
 'frequency_unusual': True,
 'velocity_unusual': False}

In [327]:
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score
)

def calculate_final_metrics(results_df, policy_name):

    actual = results_df["actual_fraud"]

    # Treat Decline as an automatic fraud decision
    predicted_fraud = (
        results_df["final_action"] == "Decline"
    ).astype(int)

    # Basic classification metrics
    accuracy = accuracy_score(actual, predicted_fraud)
    precision = precision_score(actual, predicted_fraud, zero_division=0)
    recall = recall_score(actual, predicted_fraud, zero_division=0)
    f1 = f1_score(actual, predicted_fraud, zero_division=0)

    tn, fp, fn, tp = confusion_matrix(
        actual,
        predicted_fraud
    ).ravel()

    # Customer experience
    legitimate = results_df[
        results_df["actual_fraud"] == 0
    ]

    legitimate_approval_rate = (
        legitimate["final_action"] == "Approve"
    ).mean()

    legitimate_question_rate = (
        legitimate["initial_action"] == "Question"
    ).mean()

    legitimate_examine_rate = (
        legitimate["final_action"] == "Examine"
    ).mean()

    legitimate_decline_rate = (
        legitimate["final_action"] == "Decline"
    ).mean()

    # Fraud handling
    fraud = results_df[
        results_df["actual_fraud"] == 1
    ]

    fraud_decline_rate = (
        fraud["final_action"] == "Decline"
    ).mean()

    fraud_examine_rate = (
        fraud["final_action"] == "Examine"
    ).mean()

    fraud_approved = (
        fraud["final_action"] == "Approve"
    ).sum()

    # Overall interaction
    interaction_rate = (
        results_df["initial_action"] == "Question"
    ).mean()

    # All fraud either declined or examined
    fraud_intercepted = (
        fraud["final_action"].isin(
            ["Decline", "Examine"]
        )
    ).sum()

    fraud_interception_rate = (
        fraud_intercepted / len(fraud)
    )

    metrics = {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "True Negatives": tn,
        "False Positives": fp,
        "False Negatives": fn,
        "True Positives": tp,
        "Legitimate Approval Rate": legitimate_approval_rate,
        "Legitimate Question Rate": legitimate_question_rate,
        "Legitimate Examine Rate": legitimate_examine_rate,
        "Legitimate Decline Rate": legitimate_decline_rate,
        "Fraud Automatic Decline Rate": fraud_decline_rate,
        "Fraud Examine Rate": fraud_examine_rate,
        "Fraud Approved": fraud_approved,
        "Fraud Interception Rate": fraud_interception_rate,
        "Overall Interaction Rate": interaction_rate
    }

    print("\n" + "=" * 60)
    print(policy_name)
    print("=" * 60)

    for metric, value in metrics.items():

        if isinstance(value, float):
            print(f"{metric}: {value:.4f}")

        else:
            print(f"{metric}: {value}")

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            actual,
            predicted_fraud
        )
    )

    return metrics

policy_a_metrics = calculate_final_metrics(
    test_results_df,
    "POLICY A — FINAL RESULTS"
)

policy_b_metrics = calculate_final_metrics(
    policy_b_results_df,
    "POLICY B — FINAL RESULTS"
)


POLICY A — FINAL RESULTS
Accuracy: 0.9962
Precision: 1.0000
Recall: 0.9775
F1 Score: 0.9886
True Negatives: 2000
False Positives: 0
False Negatives: 9
True Positives: 391
Legitimate Approval Rate: 0.9995
Legitimate Question Rate: 0.0270
Legitimate Examine Rate: 0.0005
Legitimate Decline Rate: 0.0000
Fraud Automatic Decline Rate: 0.9775
Fraud Examine Rate: 0.0225
Fraud Approved: 0
Fraud Interception Rate: 1.0000
Overall Interaction Rate: 0.0858

Confusion Matrix:
[[2000    0]
 [   9  391]]

POLICY B — FINAL RESULTS
Accuracy: 0.9838
Precision: 1.0000
Recall: 0.9025
F1 Score: 0.9488
True Negatives: 2000
False Positives: 0
False Negatives: 39
True Positives: 361
Legitimate Approval Rate: 0.9935
Legitimate Question Rate: 0.0210
Legitimate Examine Rate: 0.0065
Legitimate Decline Rate: 0.0000
Fraud Automatic Decline Rate: 0.9025
Fraud Examine Rate: 0.0975
Fraud Approved: 0
Fraud Interception Rate: 1.0000
Overall Interaction Rate: 0.0854

Confusion Matrix:
[[2000    0]
 [  39  361]]


In [328]:
######################### Indetifying Incorrect Decisions ###########################33
# Find Policy A's false negatives
false_negatives = test_results_df[(test_results_df["actual_fraud"] == 1) &(test_results_df["final_action"] != "Decline")]

print("False negatives:", len(false_negatives))

# Show 5 cases
print(false_negatives[[
            "transaction_id",
            "fraud_belief",
            "initial_action",
            "final_action",
            "actual_fraud"
        ]
    ].head(5)
)

False negatives: 9
    transaction_id  fraud_belief initial_action final_action  actual_fraud
501      LOC_C0997      0.619666        Examine      Examine             1
532    MERCH_C0718      0.574879        Examine      Examine             1
655     FREQ_C0834      0.619666        Examine      Examine             1
748     FREQ_C0467      0.619666        Examine      Examine             1
749      LOC_C0622      0.619666        Examine      Examine             1


In [329]:
failure_ids = [
    "LOC_C0997",
    "MERCH_C0718",
    "FREQ_C0834",
    "FREQ_C0467",
    "LOC_C0622"
]

print(
    test_df[
        test_df["transaction_id"].isin(failure_ids)
    ][
        [
            "transaction_id",
            "amount_unusual",
            "location_unusual",
            "merchant_unusual",
            "frequency_unusual",
            "velocity_unusual",
            "fraud_belief_new",
            "fraud"
        ]
    ].sort_values("transaction_id")
)

      transaction_id  amount_unusual location_unusual merchant_unusual  \
9222      FREQ_C0467           False             None            False   
9524      FREQ_C0834           False             None            False   
10102      LOC_C0622           False             None            False   
11806      LOC_C0997           False             None            False   
10513    MERCH_C0718           False             None             None   

       frequency_unusual  velocity_unusual  fraud_belief_new  fraud  
9222                True             False          0.619666      1  
9524                True             False          0.619666      1  
10102               True             False          0.619666      1  
11806               True             False          0.619666      1  
10513               True             False          0.574879      1  


In [333]:
df_trans[
    df_trans["transaction_id"] == "FREQ_C0514"
][
    [
        "transaction_id",
        "amount",
        "location",
        "merchant",
        "timestamp",
        "amount_unusual",
        "location_unusual",
        "merchant_unusual",
        "frequency_unusual",
        "velocity_unusual",
        "fraud"
    ]
]

,transaction_id,amount,location,merchant,timestamp,amount_unusual,location_unusual,merchant_unusual,frequency_unusual,velocity_unusual,fraud
9290,FREQ_C0514,2496.94,Bengaluru,Max,2026-08-10 10:25:00,False,False,False,True,False,1
